# Sentence Transformer Embeddings

Generate and save train, validation, and test embeddings for use with downstream classification models.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

W1030 15:58:46.189000 66149 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
PROJECT_ROOT = Path("../..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
EMBEDDINGS_MODEL_DIR = MODELS_DIR / "embeddings"

## Load the three existing splits

In [3]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
validation_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (58823, 6)
Validation: (12611, 6)
Test: (12632, 6)


### Verify the text and labels

In [4]:
print(train_df.columns.tolist())

display(train_df[["text", "label", "target"]].head())

['id', 'question', 'source', 'text', 'label', 'target']


,text,label,target
0,"Basically there are many categories of "" Best ...",human,0
1,"If you're hearing about it, it's because it wa...",human,0
2,"One reason is lots of catagories. However, how...",human,0
3,You can't just go around assassinating the lea...,human,0
4,It wouldn't really do any good. It's not like ...,human,0


## Load the embedding model

In [5]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding dimension:",
      embedding_model.get_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


### Verify user device

In [6]:
print("Device:", embedding_model.device)

Device: mps:0


### Small Test

In [7]:
sample_texts = train_df["text"].head(5).tolist()

sample_embeddings = embedding_model.encode(
    sample_texts,
    batch_size=32,
    show_progress_bar=True
)

print(sample_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(5, 384)


## Full encoding cells

In [8]:
X_train_embeddings = embedding_model.encode(
    train_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_validation_embeddings = embedding_model.encode(
    validation_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_test_embeddings = embedding_model.encode(
    test_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/920 [00:00<?, ?it/s]

Batches:   0%|          | 0/198 [00:00<?, ?it/s]

Batches:   0%|          | 0/198 [00:00<?, ?it/s]

In [9]:
print("Train embeddings:", X_train_embeddings.shape)
print("Validation embeddings:", X_validation_embeddings.shape)
print("Test embeddings:", X_test_embeddings.shape)

Train embeddings: (58823, 384)
Validation embeddings: (12611, 384)
Test embeddings: (12632, 384)


## Save

In [10]:
np.save(
    EMBEDDINGS_MODEL_DIR / "train_embeddings.npy",
    X_train_embeddings
)

np.save(
    EMBEDDINGS_MODEL_DIR / "validation_embeddings.npy",
    X_validation_embeddings
)

np.save(
    EMBEDDINGS_MODEL_DIR / "test_embeddings.npy",
    X_test_embeddings
)

print("Embeddings saved.")

Embeddings saved.


### Save metadata file

In [11]:
metadata = {
    "model_name": MODEL_NAME,
    "embedding_dimension": int(
        embedding_model.get_embedding_dimension()
    ),
    "train_rows": len(train_df),
    "validation_rows": len(validation_df),
    "test_rows": len(test_df),
}

metadata

import json

with open(
    EMBEDDINGS_MODEL_DIR / "embedding_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print(EMBEDDINGS_MODEL_DIR / "embedding_metadata.json")
print(EMBEDDINGS_MODEL_DIR / "test_embeddings.npy")
print(EMBEDDINGS_MODEL_DIR / "train_embeddings.npy")
print(EMBEDDINGS_MODEL_DIR / "validate_embeddings.npy")